# 02 — Automated 4-Class Dupatta Mask Bootstrap (Grounded-SAM)
**Goal:** Rapidly generate 200–300 high-precision 4-class segmentation masks (`0: other`, `1: kurta_kameez`, `2: bottom`, `3: dupatta`) on dupatta-heavy Pakistani outfits using **GroundingDINO + SAM** on Colab T4 GPU, plus side-by-side QA contact sheets for fast human verification.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers accelerate pillow numpy matplotlib tqdm

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection,
    SamModel,
    SamProcessor,
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

RESEARCH_ROOT = Path('/content/drive/MyDrive/FashionFit_PK_FYP2_Research')
DATA_DIR = RESEARCH_ROOT / '01_PK_VTON_DATASET'
MASK_OUT = DATA_DIR / 'dupatta_4class_masks' / 'masks'
QA_OUT = DATA_DIR / 'dupatta_4class_masks' / 'qa_overlays'
MASK_OUT.mkdir(parents=True, exist_ok=True)
QA_OUT.mkdir(parents=True, exist_ok=True)

# 1. Pretrained SegFormer-B2 clothes parser (anchors kurta, bottom, background, face/skin)
seg_id = 'mattmdjaga/segformer_b2_clothes'
seg_proc = SegformerImageProcessor.from_pretrained(seg_id)
seg_model = SegformerForSemanticSegmentation.from_pretrained(seg_id).to(device).eval()

# 2. GroundingDINO + SAM strictly for dupatta/shawl/scarf refinement
gd_id = 'IDEA-Research/grounding-dino-tiny'
sam_id = 'facebook/sam-vit-base'
gd_proc = AutoProcessor.from_pretrained(gd_id)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(gd_id).to(device).eval()
sam_proc = SamProcessor.from_pretrained(sam_id)
sam_model = SamModel.from_pretrained(sam_id).to(device).eval()
print('Loaded SegFormer-B2 + GroundingDINO + SAM successfully!')


In [ ]:
# Hybrid 4-Class Annotator:
#   0: other (background, face, hair, skin, shoes)
#   1: kurta_kameez (Green)  - anchored by SegFormer classes {4: Upper-clothes, 7: Dress}
#   2: bottom (Blue)         - anchored by SegFormer classes {5: Skirt, 6: Pants}
#   3: dupatta (Rose/Red)    - SegFormer class {17: Scarf} + strictly guarded GroundingDINO+SAM

PALETTE = {
    1: np.array([15, 165, 110]),   # Green: kurta_kameez
    2: np.array([59, 130, 246]),   # Blue: bottom
    3: np.array([225, 29, 72]),    # Rose/Red: dupatta
}

@torch.no_grad()
def predict_4class_mask(image: Image.Image, box_thresh=0.35, text_thresh=0.30):
    w, h = image.size
    img_area = float(w * h)

    # Step 1: Clean garment anchor from SegFormer-B2 (zero background leakage)
    seg_in = seg_proc(images=image, return_tensors='pt').to(device)
    logits = seg_model(**seg_in).logits
    upsampled = F.interpolate(logits, size=(h, w), mode='bilinear', align_corners=False)
    atr = upsampled.argmax(dim=1)[0].cpu().numpy()

    mask = np.zeros((h, w), dtype=np.uint8)
    bottom_mask = np.isin(atr, [5, 6])        # Skirt, Pants
    kurta_mask = np.isin(atr, [4, 7])         # Upper-clothes, Dress
    scarf_mask = (atr == 17)                  # Pretrained Scarf class
    protect_mask = np.isin(atr, [1, 2, 3, 9, 10, 11, 12, 13, 14, 15])  # Hair, face, skin, shoes

    mask[bottom_mask] = 2
    mask[kurta_mask] = 1
    mask[scarf_mask] = 3

    # Step 2: Detect draped dupattas / shawls with GroundingDINO + SAM (best IoU mask only)
    prompt = 'dupatta . shawl . scarf . stole .'
    inputs = gd_proc(images=image, text=prompt, return_tensors='pt').to(device)
    outputs = gd_model(**inputs)
    try:
        results = gd_proc.post_process_grounded_object_detection(
            outputs, inputs.input_ids, threshold=box_thresh, text_threshold=text_thresh, target_sizes=[(h, w)]
        )[0]
    except TypeError:
        results = gd_proc.post_process_grounded_object_detection(
            outputs, inputs.input_ids, box_threshold=box_thresh, text_threshold=text_thresh, target_sizes=[(h, w)]
        )[0]

    boxes = results['boxes']
    if len(boxes) > 0:
        valid_boxes = []
        for b in boxes.cpu().tolist():
            x1, y1, x2, y2 = b
            box_area_ratio = ((x2 - x1) * (y2 - y1)) / img_area
            if 0.01 < box_area_ratio < 0.60:
                valid_boxes.append(b)

        if len(valid_boxes) > 0:
            sam_in = sam_proc(image, input_boxes=[valid_boxes], return_tensors='pt').to(device)
            sam_out = sam_model(**sam_in)
            masks = sam_proc.image_processor.post_process_masks(
                sam_out.pred_masks.cpu(), sam_in['original_sizes'].cpu(), sam_in['reshaped_input_sizes'].cpu()
            )[0]
            iou_scores = sam_out.iou_scores.cpu()[0]  # shape: [num_boxes, 3]

            kurta_area = max(kurta_mask.sum(), 1)
            for i, m_tensor in enumerate(masks):
                best_idx = int(iou_scores[i].argmax().item())
                cand = m_tensor[best_idx].numpy().astype(bool)
                cand = cand & (~protect_mask)

                cand_ratio = cand.sum() / img_area
                # Reject background leaks (>42% of image) or tiny noise (<0.4%)
                if cand_ratio < 0.004 or cand_ratio > 0.42:
                    continue
                # Reject false positives that simply re-segment the entire kurta (>75% overlap with kurta)
                kurta_overlap = (cand & kurta_mask).sum() / kurta_area
                if kurta_overlap > 0.75:
                    continue

                mask[cand] = 3

    return mask

def make_overlay(image: Image.Image, mask: np.ndarray):
    arr = np.array(image).astype(np.float32)
    out = arr.copy()
    for cls_id, rgb in PALETTE.items():
        m = mask == cls_id
        out[m] = 0.55 * arr[m] + 0.45 * rgb
    return Image.fromarray(out.astype(np.uint8))

manifest = pd.read_csv(DATA_DIR / 'pk_vton_manifest.csv')
MAX_SAMPLES = min(300, len(manifest))
print(f'Re-annotating {MAX_SAMPLES} Pakistani images with Hybrid SegFormer + Guarded Grounded-SAM...')

stats = []
for _, row in tqdm(manifest.head(MAX_SAMPLES).iterrows(), total=MAX_SAMPLES):
    img_id = row['image_id']
    p = Path(row['path'])
    if not p.exists():
        continue
    im = Image.open(p).convert('RGB')
    m = predict_4class_mask(im)
    Image.fromarray(m).save(MASK_OUT / f'{img_id}.png')
    make_overlay(im, m).save(QA_OUT / f'{img_id}_qa.jpg', quality=85)
    stats.append({
        'image_id': img_id,
        'image_path': str(p),
        'mask_path': str(MASK_OUT / f'{img_id}.png'),
        'has_dupatta': bool((m == 3).sum() > 800),
        'dupatta_pct': round(100.0 * (m == 3).mean(), 2),
        'kurta_pct': round(100.0 * (m == 1).mean(), 2),
        'bottom_pct': round(100.0 * (m == 2).mean(), 2),
    })

qa_df = pd.DataFrame(stats)
qa_df.to_csv(DATA_DIR / 'dupatta_4class_masks' / 'auto_mask_qa_index.csv', index=False)
print('Done!')
print('Dupatta-positive images:', int(qa_df['has_dupatta'].sum()), '/', len(qa_df))
print('Mean % -> Kurta:', round(qa_df['kurta_pct'].mean(), 2), '| Bottom:', round(qa_df['bottom_pct'].mean(), 2), '| Dupatta:', round(qa_df['dupatta_pct'].mean(), 2))
